# Phase 2 — Liveness (Anti-Spoofing) Training on Colab
Binary classifier **Live (1) vs Spoof (0)** on face crops → ONNX → .

**Dataset :** CASIA Face Anti-Spoofing (compact, ~300 Mo - 1 Go) téléchargé automatiquement via KaggleHub.
**Modèle :** MobileNetV3-Small optimisé pour une inférence ultra-rapide en temps réel.
**Runtime Colab recommandé :** GPU T4 (ou CPU).

## 1. Setup & Dépendances

In [ ]:
!pip -q install insightface onnx onnxruntime kagglehub

import math, random, json, time, os, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Choix du stockage : False pour travailler directement et rapidement sur le disque local de Colab
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/face-access-control')
else:
    ROOT = Path('/content/face-access-control')

LIVE_DIR = ROOT / 'data/liveness'
RES_DIR = ROOT / 'evaluation/results'
OUT_DIR = ROOT / 'models/liveness'
for d in (LIVE_DIR, RES_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SIZE, SCALE = 128, 2.0          # Doit correspondre a config.yaml (size / crop_scale)
TRAIN_SOURCES = ['casia']       # source d'entrainement
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_CUDA = DEVICE == 'cuda'
print(f"Dispositif d'entrainement : {DEVICE}")

## 2. Recadrage facial contextuel et Détecteur InsightFace

In [ ]:
def crop_face(frame_bgr, bbox, scale=2.0, size=128):
    """Square crop centre sur le visage avec facteur scale=2.0 (garde le contexte des bords d'ecran et papier).
    Identique au code d'inference de l'application locale."""
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in bbox]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(x2 - x1, y2 - y1) * scale / 2
    l, t = int(round(cx - half)), int(round(cy - half))
    r, b = int(round(cx + half)), int(round(cy + half))
    crop = frame_bgr[max(t, 0):min(b, h), max(l, 0):min(r, w)]
    if crop.size == 0:
        return np.zeros((size, size, 3), np.uint8)
    crop = cv2.copyMakeBorder(crop, max(0, -t), max(0, b - h), max(0, -l), max(0, r - w), cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

from insightface.app import FaceAnalysis
det = FaceAnalysis(name='buffalo_s', root='/content/insightface', allowed_modules=['detection'],
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
det.prepare(ctx_id=0 if USE_CUDA else -1, det_size=(640, 640))
print("Detecteur facial initialise.")

## 3. Téléchargement et Ingestion du Dataset CASIA

In [ ]:
import kagglehub

def download_casia():
    """Telecharge le dataset CASIA Face Anti-Spoofing via KaggleHub."""
    print("Telechargement du dataset CASIA...")
    try:
        path = kagglehub.dataset_download("tapakah68/casia-face-antispofing")
    except Exception:
        path = kagglehub.dataset_download("fabriziocannizzo/casia-fasd-face-antispoofing")
    print("Dataset telecharge dans :", path)
    return Path(path)

def ingest_images_from_disk(file_paths, source, label, group_name):
    out_dir = LIVE_DIR / source / label / group_name
    out_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for p in file_paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        faces = det.get(img)
        if not faces:
            continue
        f = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        if f.det_score < 0.6:
            continue
        crop = crop_face(img, f.bbox, SCALE, SIZE)
        out_file = out_dir / f"{Path(p).stem}_{count:04d}.jpg"
        cv2.imwrite(str(out_file), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        count += 1
    return count

def ingest_casia_dataset(root_path, max_images_per_class=1200):
    root_path = Path(root_path)
    print(f"Indexation du dataset CASIA depuis {root_path}...")
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    all_files = [p for p in root_path.rglob("*") if p.suffix.lower() in img_exts]
    print(f"Total d'images trouvees : {len(all_files)}")
    
    real_files, spoof_files = [], []
    for p in all_files:
        path_str = str(p).lower()
        is_real = any(k in path_str for k in ["/real/", "/live/", "\real\", "\live\", "genuine"])
        is_spoof = any(k in path_str for k in ["/fake/", "/spoof/", "\fake\", "\spoof\", "attack"])
        
        if not is_real and not is_spoof:
            stem = p.stem.lower()
            if stem in ["1", "2", "hr_1"] or stem.startswith("1_") or stem.startswith("2_"):
                is_real = True
            elif any(stem.startswith(f"{i}") for i in range(3, 9)) or "hr_" in stem:
                is_spoof = True
                
        if is_real:
            real_files.append(p)
        elif is_spoof:
            spoof_files.append(p)
            
    print(f"Images Real (Live) identifiees : {len(real_files)} | Images Spoof identifiees : {len(spoof_files)}")
    
    random.seed(42)
    random.shuffle(real_files)
    random.shuffle(spoof_files)
    real_sample = real_files[:max_images_per_class]
    spoof_sample = spoof_files[:max_images_per_class]
    
    n_groups = 5
    for i in range(n_groups):
        g_real = real_sample[i::n_groups]
        g_spoof = spoof_sample[i::n_groups]
        n_r = ingest_images_from_disk(g_real, "casia", "live", f"casia_live_g{i+1}")
        n_s = ingest_images_from_disk(g_spoof, "casia", "spoof", f"casia_spoof_g{i+1}")
        print(f"  Groupe {i+1} : {n_r} live, {n_s} spoof ingeres.")

# Lancement du telechargement et de l'ingestion de CASIA
casia_path = download_casia()
ingest_casia_dataset(casia_path, max_images_per_class=1000)

## 4. Indexation et Partitionnement (Train / Val / Test)

In [ ]:
def index_dataset(root):
    rows = []
    for src in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        for lab_name, lab in (('live', 1), ('spoof', 0)):
            target = src / lab_name
            if not target.exists():
                continue
            for gdir in sorted(target.glob('*')):
                if not gdir.is_dir():
                    continue
                for p in sorted(gdir.glob('*.jpg')):
                    rows.append((src.name, lab, gdir.name, f'{src.name}/{gdir.name}', str(p)))
    return pd.DataFrame(rows, columns=['source', 'label', 'group', 'key', 'path'])

def assign_splits(df, train_sources, val=0.15, test=0.2, seed=0):
    rng = np.random.default_rng(seed)
    split = {}
    for src in train_sources:
        d = df[df.source == src]
        kl = d.groupby('key')['label'].agg(lambda s: tuple(sorted(set(s)))).to_dict()
        for lset in sorted(set(kl.values())):
            ks = sorted(k for k, v in kl.items() if v == lset)
            rng.shuffle(ks)
            n = len(ks)
            if n < 3:
                for k in ks:
                    split[k] = 'train'
                continue
            n_test, n_val = max(1, round(n * test)), max(1, round(n * val))
            for i, k in enumerate(ks):
                split[k] = 'test' if i < n_test else ('val' if i < n_test + n_val else 'train')
    df = df.copy()
    df['split'] = df['key'].map(split).fillna('train')
    return df

df = index_dataset(LIVE_DIR)
assert len(df) > 0, f"Aucune image trouvee sous {LIVE_DIR}."
df = assign_splits(df, TRAIN_SOURCES)

d_train = df[df.split == 'train']
d_val = df[df.split == 'val']
if len(d_val) == 0 or d_val.label.nunique() < 2:
    print("Utilisation du partitionnement stratifie alternatif...")
    d_train, d_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df.label)
    d_test = d_val.copy()
else:
    d_test = df[df.split == 'test']

print(f"Train: {len(d_train)} images | Val: {len(d_val)} images | Test: {len(d_test)} images")
print(d_train['label'].value_counts().rename({1: 'live', 0: 'spoof'}))

## 5. Entraînement du Modèle MobileNetV3-Small

In [ ]:
TF_TRAIN = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.RandomApply([T.GaussianBlur(3, (0.1, 1.2))], p=0.2),
    T.ToTensor(),
])
TF_EVAL = T.ToTensor()

class DS(Dataset):
    def __init__(self, d, train):
        self.p, self.y, self.tf = d.path.tolist(), d.label.tolist(), (TF_TRAIN if train else TF_EVAL)
    def __len__(self):
        return len(self.p)
    def __getitem__(self, i):
        img = Image.open(self.p[i]).convert('RGB')
        if img.size != (SIZE, SIZE):
            img = img.resize((SIZE, SIZE))
        return self.tf(img), self.y[i]

class AntiSpoofNet(nn.Module):
    """MobileNetV3-small binaire (Live vs Spoof). Normalisation integree au reseau."""
    def __init__(self, pretrained=True):
        super().__init__()
        w = torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        m = torchvision.models.mobilenet_v3_small(weights=w)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
        self.net = m
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.net((x - self.mean) / self.std)

@torch.no_grad()
def predict(model, d, bs=64):
    model.eval()
    out = []
    loader = DataLoader(DS(d, False), batch_size=bs, shuffle=False)
    for x, _ in loader:
        x = x.to(DEVICE)
        logits = model(x)
        out.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def train(d_train, d_val, epochs=12, bs=64, lr=3e-4):
    model = AntiSpoofNet(pretrained=True).to(DEVICE)
    counts = np.bincount(d_train.label, minlength=2)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(len(d_train) / (2 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(DS(d_train, True), batch_size=bs, shuffle=True, drop_last=len(d_train) > bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(loader))
    best_auc, best_state = -1.0, None
    
    print(f"Lancement de l'entrainement sur {epochs} epoques ({DEVICE})...")
    for ep in range(epochs):
        model.train()
        tot, t0 = 0.0, time.time()
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item() * len(x)
            
        s = predict(model, d_val)
        auc = roc_auc_score(d_val.label, s)
        pred_live = s >= 0.5
        acc = (pred_live == d_val.label.values).mean()
        print(f"Epoque {ep+1:02d}/{epochs:02d} | Perte: {tot/len(d_train):.4f} | Val AUC: {auc:.4f} | Precision: {acc*100:.1f}% | {time.time()-t0:.1f}s")
        if auc > best_auc:
            best_auc, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            
    model.load_state_dict(best_state)
    return model

model = train(d_train, d_val, epochs=12, bs=64)

## 6. Calibration du Seuil EER et Évaluation

In [ ]:
def eer_threshold(scores, labels):
    fpr, tpr, thr = roc_curve(labels, scores)
    i = int(np.nanargmin(np.abs(fpr - (1 - tpr))))
    return float(thr[i]) if np.isfinite(thr[i]) else 0.5

s_val = predict(model, d_val)
THR = eer_threshold(s_val, d_val.label.values)
print(f"
Seuil optimal calcule (EER) = {THR:.4f}")

s_test = predict(model, d_test)
test_auc = roc_auc_score(d_test.label, s_test)
pred_test = s_test >= THR
apcer = float(pred_test[d_test.label.values == 0].mean())  # spoof accepte
bpcer = float((~pred_test[d_test.label.values == 1]).mean()) # live rejete
acer = (apcer + bpcer) / 2

print(f"
Resultats sur Test Set (Seuil = {THR:.3f}) :")
print(f"   AUC    : {test_auc:.4f}")
print(f"   APCER  : {apcer*100:.2f}% (Taux de faux acceptes)")
print(f"   BPCER  : {bpcer*100:.2f}% (Taux de faux rejetes)")
print(f"   ACER   : {acer*100:.2f}% (Erreur moyenne)")

fpr, tpr, _ = roc_curve(d_test.label, s_test)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'Test ROC (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('APCER (spoof accepte)')
plt.ylabel('1 - BPCER (live accepte)')
plt.title('Courbe ROC Anti-Spoofing (CASIA)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig(RES_DIR / 'liveness_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exportation en ONNX et Téléchargement Automatique

In [ ]:
class Export(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, x):
        return torch.softmax(self.m(x), dim=1)[:, 1]   # P(live)

net = Export(model.to('cpu').eval())
dummy = torch.rand(1, 3, SIZE, SIZE)
onnx_path = OUT_DIR / 'antispoof.onnx'
json_path = OUT_DIR / 'antispoof.json'

kw = dict(input_names=['input'], output_names=['prob_live'], dynamic_axes={'input': {0: 'batch'}, 'prob_live': {0: 'batch'}}, opset_version=17)
torch.onnx.export(net, dummy, str(onnx_path), **kw)

meta = {
    "size": SIZE,
    "scale": SCALE,
    "threshold": round(THR, 4),
    "output": "prob_live (1 = live)",
    "dataset": "CASIA Face Anti-Spoofing",
    "test_auc": round(test_auc, 4),
    "acer": round(acer, 4),
    "exported_at": time.strftime("%Y-%m-%d %H:%M:%S")
}
json_path.write_text(json.dumps(meta, indent=2))
print(f"Modele exporte : {onnx_path}")
print(f"Metadonnees exportees : {json_path}")

from google.colab import files
print("Telechargement automatique de antispoof.onnx et antispoof.json...")
files.download(str(onnx_path))
files.download(str(json_path))